In [ ]:
import sys
from pathlib import Path
import torch
sys.path.insert(0, str(Path.cwd().resolve().parents[1]))



from transformers import AutoTokenizer, BitsAndBytesConfig
from transformers import AutoTokenizer, AutoModelForCausalLM

from MoE_Optimization.common.clustering_utils import get_qwen


MODEL = "Qwen/Qwen1.5-MoE-A2.7B"


quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,

    bnb_4bit_compute_dtype=torch.float16,  # Compute in float16 for speed
    bnb_4bit_use_double_quant=True,  # Double quantization for extra memory savings
    # Normalized float 4-bit (optimal for LLMs)
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True,
                                            device_map='cuda:2',
                                            )
model = AutoModelForCausalLM.from_pretrained(MODEL, trust_remote_code=True,
                                            device_map='cuda:2',
                                            dtype=torch.float16,
                                            # quantization_config=quantization_config,
                                            cache_dir="../../.cache"
                                            )

I0000 00:00:1780147190.152319   55424 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1780147190.217152   55424 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780147191.738062   55424 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 12.00 MiB. GPU 2 has a total capacity of 39.38 GiB of which 1.38 MiB is free. Process 641577 has 39.37 GiB memory in use. Of the allocated memory 38.82 GiB is allocated by PyTorch, and 72.73 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [2]:
from MoE_Optimization.common.expert_utils.expert_clusterer_pipeline import ExpertClustererPipeline
expertClusterer = ExpertClustererPipeline(model, tokenizer)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cais/mmlu' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Initialized MMLU batch generator with 5 subjects
Split: validation, Batch size: 16

📚 Loading subject 1/5: anatomy


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cais/mmlu' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  ✓ Created batch with 14 questions

📚 Loading subject 2/5: machine_learning


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cais/mmlu' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  ✓ Created batch with 11 questions

📚 Loading subject 3/5: management


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cais/mmlu' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  ✓ Created batch with 11 questions

📚 Loading subject 4/5: philosophy


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cais/mmlu' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  ✓ Created batch with 16 questions

📚 Loading subject 5/5: world_religions
  ✓ Created batch with 16 questions


In [3]:
from MoE_Optimization.common.expert_utils.expert_clusterer_pipeline import ONE_LAYER_CLUSTERING_METHOD
ONE_LAYER_CLUSTERING_METHOD

['create_clusters_by_PI',
 'create_clusters_by_EC',
 'create_clusters_by_PI_but_find_max_on_EC',
 'create_clusters_by_EC_but_find_min_on_PI',
 'create_clusters_by_PI_objective_and_EC_product',
 'create_clusters_by_EC_objective_and_PI_product',
 'create_clusters_by_PI_greedy_aglomerative',
 'create_clusters_by_EC_greedy_aglomerative',
 'create_clusters_by_PI_front_points_and_Calinski_metric']

In [4]:
from MoE_Optimization.common.moe_utils import discover_moe_layers

layers = discover_moe_layers(model)
layers = [layers[0], layers[23]]

max_cluster_size=2 will work only for supported algorithms

In [5]:
results = []
for method in ONE_LAYER_CLUSTERING_METHOD:
    clustering = expertClusterer.match_experts(layers, method, max_cluster_size=2)
    print(method, clustering)
    results.append(clustering)

create_clusters_by_PI {0: [[22, 23], [12, 27], [49, 13], [48, 32], [37, 39], [17, 28], [44, 29], [0, 40], [41, 11], [18, 43], [19, 24], [45, 55], [9, 16], [6, 34], [36, 15], [53, 58], [51, 25], [10, 59], [31, 54], [56, 21], [14, 30], [35, 42], [8, 7], [2, 3], [50, 38], [1], [5, 20], [33], [57, 46], [4], [26, 52], [47]], 23: [[43, 59], [30, 49], [47, 19], [14, 7], [28, 48], [29, 1], [41, 11], [3, 37], [24, 26], [22, 56], [53, 32], [51, 35], [44, 40], [57, 23], [15, 21], [39, 16], [4, 42], [31, 34], [50, 38], [25, 13], [54, 52], [5, 6], [0, 17], [55, 58], [33, 9], [10], [27, 20], [8], [2, 18], [12], [45, 46], [36]]}
create_clusters_by_EC {0: [[30, 27], [40, 22], [39, 50], [20, 3], [36, 10], [11, 26], [4, 37], [57, 54], [7, 9], [17, 58], [24, 42], [29, 15], [51, 45], [41, 48], [2, 47], [8, 53], [34, 52], [23, 12], [55, 16], [35, 59], [49, 13], [5, 14], [28, 6], [33, 19], [43, 31], [18], [1, 25], [38], [44, 0], [32], [21, 46], [56]], 23: [[49, 20], [39, 57], [0, 35], [55, 42], [13, 4], [59

In [5]:
for method in ONE_LAYER_CLUSTERING_METHOD:
    clustering = expertClusterer.match_experts(layers, method)
    print(method, clustering)

create_clusters_by_PI {0: [[22, 23], [12, 27], [50, 38, 1], [5, 20, 33], [49, 13], [48, 32], [37, 39], [17, 28], [57, 46, 4], [44, 29], [0, 40], [26, 52, 47], [41, 11], [18, 43], [19, 24], [45, 55], [9, 16], [6, 34], [36, 15], [53, 58], [51, 25], [10, 59], [31, 54], [56, 21], [14, 30], [35, 42], [8, 7], [2, 3]], 23: [[33, 9, 10], [27, 20, 8], [2, 18, 12], [43, 59], [30, 49], [47, 19], [14, 7], [45, 46, 36], [28, 48], [29, 1], [41, 11], [3, 37], [24, 26], [22, 56], [53, 32], [51, 35], [44, 40], [57, 23], [15, 21], [39, 16], [4, 42], [31, 34], [50, 38], [25, 13], [54, 52], [5, 6], [0, 17], [55, 58]]}
create_clusters_by_EC {0: [[30, 27], [40, 22], [39, 50], [20, 3], [36, 10], [11, 26], [4, 37], [57, 54], [7, 9], [17, 58], [43, 31, 18], [24, 42], [29, 15], [1, 25, 38], [51, 45], [41, 48], [2, 47], [8, 53], [44, 0, 32], [34, 52], [23, 12], [21, 46, 56], [55, 16], [35, 59], [49, 13], [5, 14], [28, 6], [33, 19]], 23: [[22, 18, 6], [49, 20], [39, 57], [0, 35], [55, 42], [13, 4], [59, 48], [23,